# Google Colab Pro (`NVIDIA L4 24GB` — Guarded Budget) · Unified 10-Class Multi-Product Detector Training

Trains **`YOLOv11-m`**, **`RT-DETR-Large`**, and/or **`YOLOv9-c`** on the **Unified 10-Class Multi-Product PCB/PCBA Dataset** with all 4 credit protections enabled:
1. **L4 24GB Sweet Spot (`~4.5 CU/hr`)**: ~3x better epochs-per-credit efficiency than A100.
2. **Local NVMe SSD Staging (`/content/datasets`)**: Extracts `.tar` or pulls from Kaggle API in `<60s` so training never stalls on Google Drive FUSE I/O.
3. **Periodic Epoch Checkpoint Sync**: Mirrors `best.pt` and `last.pt` to `/content/drive/MyDrive/PCB_FYP_Experiments/weights/` every 10 epochs.
4. **Mandatory `runtime.unassign()`**: Automatically terminates the Colab VM the instant training finishes.

In [ ]:
!pip install -q ultralytics>=8.3.40 kaggle albumentations onnx onnxruntime-gpu

import json, os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/Darren7218/PCB-Defect-Detection-FYP.git'
BRANCH   = 'feat/multiproduct-patchcore-v2'
REPO_DIR = Path('/content/PCB-Defect-Detection-FYP')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

In [ ]:
# Configure which architectures to train in this session
# Estimated L4 burn: yolo11m (~16-20 CU), yolov9c (~18-22 CU), rtdetr_l (~22-26 CU)
MODELS_TO_TRAIN   = ['yolo11m', 'rtdetr_l', 'yolov9c']  # Edit list to run a subset
EPOCHS            = 100
IMGSZ             = 640
AUTO_UNASSIGN_VM  = True   # Keep True to guarantee zero idle credit burn after completion!

# Optional: Kaggle Dataset slug if you published `unified_pcb_10cls` from Kaggle Notebook 01
KAGGLE_UNIFIED_SLUG = ''   # e.g. 'darrenooh/unified-pcb-10cls' (leave empty to build/stage from Drive + Git)

In [ ]:
from ultralytics import RTDETR, YOLO
from src.cloud.colab_guard import (
    ColabCreditGuard,
    mount_google_drive,
    stage_archive_to_local_ssd,
    stage_kaggle_dataset_to_ssd,
    sync_artifacts_to_drive,
)

drive_root = mount_google_drive()
exp_root = (drive_root / 'PCB_FYP_Experiments') if drive_root else Path('/content/PCB_FYP_Experiments')
weights_drive_dir = exp_root / 'weights'
weights_drive_dir.mkdir(parents=True, exist_ok=True)

# Configure Kaggle credentials if stored on Drive
if (exp_root / 'kaggle.json').exists():
    kdst = Path.home() / '.kaggle' / 'kaggle.json'
    kdst.parent.mkdir(parents=True, exist_ok=True)
    kdst.write_bytes((exp_root / 'kaggle.json').read_bytes())
    os.chmod(kdst, 0o600)

with ColabCreditGuard(
    job_name='multiproduct_10cls_detectors_L4',
    drive_sync_dir=exp_root / 'runs_backup',
    max_cu_budget=75.0,
    auto_unassign=AUTO_UNASSIGN_VM,
) as guard:

    # 1. Stage dataset onto fast local NVMe SSD (/content/datasets)
    ssd_root = Path('/content/datasets')
    tar_on_drive = exp_root / 'datasets' / 'unified_pcb_10cls.tar'
    if tar_on_drive.exists():
        staged = stage_archive_to_local_ssd(tar_on_drive, ssd_root)
        data_yaml = next(staged.rglob('data.yaml'))
    elif KAGGLE_UNIFIED_SLUG:
        staged = stage_kaggle_dataset_to_ssd(KAGGLE_UNIFIED_SLUG, ssd_root)
        data_yaml = next(staged.rglob('data.yaml'))
    else:
        # Build directly onto local NVMe SSD (auto-clones DeepPCB + ingests any staged VOC_PCB / VisA / PCBA)
        voc_dir = stage_kaggle_dataset_to_ssd('darrenooh/yolo-ready', ssd_root) if (Path.home()/'.kaggle'/'kaggle.json').exists() else None
        cmd = [sys.executable, '-m', 'src.datasets.prepare_multiproduct_pcb', '--out-dir', str(ssd_root / 'built'), '--download-deeppcb', '--download-hf-pcba']
        if voc_dir:
            v_sub = next(voc_dir.rglob('VOC_PCB'), voc_dir)
            cmd += ['--voc-pcb', str(v_sub)]
        subprocess.run(cmd, check=True)
        data_yaml = ssd_root / 'built' / 'unified_pcb_10cls' / 'data.yaml'

    # Rewrite `path:` inside data.yaml to the exact local SSD path
    yaml_txt = data_yaml.read_text()
    lines = [f'path: {data_yaml.parent.as_posix()}' if l.startswith('path:') else l for l in yaml_txt.splitlines()]
    data_yaml.write_text('\n'.join(lines) + '\n')
    print('Training on local NVMe SSD data.yaml:', data_yaml)

    # 2. Periodic Drive sync callback every 10 epochs
    def make_sync_cb(export_name: str):
        def _on_epoch_end(trainer):
            ep = trainer.epoch + 1
            if ep % 10 == 0 or ep == trainer.epochs:
                best_p = Path(trainer.save_dir) / 'weights' / 'best.pt'
                if best_p.exists():
                    shutil.copy2(best_p, weights_drive_dir / export_name)
                    print(f'[DriveSync] Epoch {ep}: synced {export_name}')
        return _on_epoch_end

    metrics_summary = {}

    for arch in MODELS_TO_TRAIN:
        print('\n' + '=' * 68)
        print(f'Launching {arch} for {EPOCHS} epochs on L4...')
        if arch == 'yolo11m':
            model = YOLO('yolo11m.pt')
            out_pt = 'yolo11m_multiproduct_best.pt'
            train_kw = dict(batch=32, optimizer='AdamW', lr0=1e-3, cos_lr=True)
        elif arch == 'yolov9c':
            model = YOLO('yolov9c.pt')
            out_pt = 'yolov9_multiproduct_best.pt'
            train_kw = dict(batch=24, optimizer='SGD', lr0=1e-2, cos_lr=True)
        elif arch == 'rtdetr_l':
            model = RTDETR('rtdetr-l.pt')
            out_pt = 'rtdetr_multiproduct_best.pt'
            # AdamW + lower lr0 stabilizes Vision Transformer self-attention matrices
            train_kw = dict(batch=16, optimizer='AdamW', lr0=1e-4, lrf=0.05, warmup_epochs=5, cos_lr=True)
        else:
            raise ValueError(f'Unknown arch: {arch}')

        model.add_callback('on_fit_epoch_end', make_sync_cb(out_pt))
        res = model.train(
            data=str(data_yaml),
            epochs=EPOCHS,
            imgsz=IMGSZ,
            patience=20,
            amp=True,
            workers=4,
            project='/content/runs_multiproduct',
            name=arch,
            exist_ok=True,
            **train_kw,
        )
        best_ckpt = Path(res.save_dir) / 'weights' / 'best.pt'
        if best_ckpt.exists():
            shutil.copy2(best_ckpt, weights_drive_dir / out_pt)
            guard.register_artifact(weights_drive_dir / out_pt)

        # Blind Test Split Evaluation
        val_res = model.val(data=str(data_yaml), split='test', imgsz=IMGSZ)
        metrics_summary[arch] = {
            'mAP50': float(val_res.box.map50),
            'mAP50_95': float(val_res.box.map),
            'precision': float(val_res.box.mp),
            'recall': float(val_res.box.mr),
            'checkpoint': out_pt,
        }
        (weights_drive_dir / 'multiproduct_detector_metrics.json').write_text(json.dumps(metrics_summary, indent=2))
        print(f'{arch} Test Results:', metrics_summary[arch])